# III) Single material optimization


## 1) Geometry

First, we define the geometry, material distribution, and generate the mesh.

In [35]:
pole_pairs = 6
bundles_per_half_slot = 5

# Generate the corresponding mesh
from utils.geometry import machine_mesh
mesh = machine_mesh(p=pole_pairs, 
                    bundles_per_half_slot=bundles_per_half_slot, 
                    hBundle=0.25e-3
                    )

print(f"Generated mesh with {mesh.nv} nodes, {mesh.ne} elements")

Generated mesh with 15477 nodes, 30870 elements


________
## 2) Setup of the physical problem

See notebook [I_Reference_simulation](I_Reference_simulation.ipynb).

### 2.a) Material properties

In [36]:
mur_iron = 1000           # Relative magnetic permeability of iron core
sigma_copper = 5.8e7      # Copper conductivity (S/m)
Br = 1                    # Remanent flux density of magnets (T)


# Define space-varying material coefficients
from ngsolve import pi
mu0 = 4e-7 * pi
nu = mesh.MaterialCF({"core_stator" : 1 / (mu0 * mur_iron)}, default = 1/mu0)     # reluctivity
sigma = mesh.MaterialCF({"slot(.*)_bundle.*" : sigma_copper})                     # conductivity
from utils.physics import magnetization_halbach
Mcplx = mesh.MaterialCF({"rotor" : magnetization_halbach(br = Br, p=pole_pairs)}) # magnetization

### 2.b) Current supply

In [37]:
freq = 1000 # Electrical frequency (Hz)
Jrms = 10e6 # Current density (A/m²)
phi = 150   # load angle (°), chosen to maximize average torque
winding_type = "distributed" # "distributed" or "concentrated"

from ngsolve import Integrate
from utils.supply import phase_current, winding_arrangement, bundle_arrangement

S_bundle = Integrate(1, mesh.Materials("slot11_bundle0"))
Irms = Jrms * S_bundle
phase = phase_current(I_rms=Irms,  load_angle=phi*pi/180)
winding = winding_arrangement(phase, type = winding_type)
bundles = bundle_arrangement(winding, bundles_per_half_slot=bundles_per_half_slot)

### 2.c) Finite element space

In [38]:
curve_order = 2
fem_order = curve_order

from ngsolve import H1, Periodic
fes = Periodic( H1(mesh.Curve(curve_order), 
                   order = fem_order, 
                   dirichlet =  "shaft|out", 
                   complex = True),  [-1]*7 )
print(f"Number of degrees of freedom of the FE space: {fes.ndof}")

Number of degrees of freedom of the FE space: 61823


_________
## 3) Optimization of each bundle material

Too high conductivity leads to high AC losses, while too low conductivity leads to high DC losses. There is an optimum point that can be found. An optimization of the whole conductivity of a single material coil was conducted in [III_single_material_optimization](III_single_material_optimization.ipynb).
However, hairpin winding may enable a different conductivity per bundle.
The dimension of the optimization problem is now too big to apply an exhaustive search!

### 3.a) Setup of the optimization problem

In [47]:
# Problem parameters
sigma0 = sigma_copper         # initial conductivity
sigma_min = sigma_copper/10   # minimum admissible conductivity
sigma_max = sigma_copper      # maximum admissible conductivity

# Conductivity space
from ngsolve import L2, GridFunction
fes_sigma = L2(mesh, definedon = "slot(.*)_bundle.*")
x0 = GridFunction(fes_sigma)
x0.Set(sigma0)

In [ ]:
# Definition of the functions of interest

from utils.physics import solve_magnetoharmonic, joule_losses
def state_function(conductivity):
    """ Returns the solution of the magnetoharmonic problem for a given conductivity value """
    result = solve_magnetoharmonic(fes=fes,
                                   reluctivity=nu,
                                   conductivity=conductivity,
                                   magnetization=Mcplx,
                                   frequency=freq,
                                   supply=bundles
                                   )
    return result

def objective_function(result):
    """ Total Joule losses to minimize """
    return joule_losses(result)

In [ ]:
# Algorithm parameters

from copy import copy
from numpy import sign

def descent(grad):
    """ Extract descent direction from the gradient """
    descent = copy(grad)
    descent.vec.data = - 1e7 * sign(grad.vec)
    return descent

# Derivative
from utils.optimization import d_joule_losses
from ngsolve import InnerProduct, CoefficientFunction as CF

def grad_joule_losses_winding(result):
    """ Compute the gradient of the objective function w.r.t the conductivity value"""
    dx = GridFunction(result["info"]["conductivity"].space)
    dx.Set(1)
    df = d_joule_losses(result, dx) # directional derivative of the objective function in the direction dx
    # we want now to provide a conductivity field representing the steepest admissible ascent direction (we call this the "gradient")
    dfdx = InnerProduct(df.vec, dx.vec) # derivative of f in the direction unit perturbation dx
    grad = copy(dx)
    grad.vec.data = dfdx * dx.vec  # dx is an admissible unit perturbation of the conductivity
    return grad

# Derivative
def grad_joule_losses_bundle(result):
    """ Compute the gradient of the objective function w.r.t the conductivity value of each bundle """
    fes_sigma = result["info"]["conductivity"].space
    dx_global = GridFunction(fes_sigma)
    dx_global.Set(1)
    df = d_joule_losses(result, dx_global)
    # we want now to provide a conductivity field representing the steepest admissible ascent direction (we call this the "gradient")
    dfdx = CF(0)
    for bundle in result["info"]["supply"].keys():
        dx = GridFunction(fes_sigma)
        dx.Set(mesh.MaterialCF({bundle : 1})) # dx is a unit perturbation of sigma in a single bundle
        dfdx += InnerProduct(df.vec, dx.vec) * copy(dx)  
    grad_f = GridFunction(fes_sigma)
    grad_f.Set(dfdx.Compile())
    return grad_f


from utils.optimization import gradient_descent

results_optim_global_sigma = gradient_descent(state=state_function,
                                 objective = objective_function,
                                 d_objective = grad_joule_losses_winding,
                                 x0 = x0,
                                 x_min = sigma_min,
                                 x_max = sigma_max,
                                 descent = descent)

WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'Objects': {'Wireframe': Fal…

it 1 ✅| obj = 3.383e+04 | step = 1.00e+00 | crit = 1.00e+00
it 2 ✅| obj = 3.087e+04 | step = 1.20e+00 | crit = 1.37e+00
it 3 ✅| obj = 2.604e+04 | step = 1.44e+00 | crit = 1.84e+00
it 4 ✅| obj = 1.868e+04 | step = 1.73e+00 | crit = 2.15e+00


In [ ]:
# Display the optimal conductivity distribution
from numpy import nan
from ngsolve.webgui import Draw
result_optim_global = results_optim_global_sigma["solution"][-1] + mesh.MaterialCF({"slot.*_bundle.*" : 0}, default = nan)
Draw(result_optim_global, results_optim_global_sigma["solution"][-1].space.mesh,
     settings = {"Objects" : {"Wireframe" : False}, "Colormap" : {"ncolors" : 32}},
     filename = "scenes/optim/sigma_optim_global.html",
     min = sigma_min, max = sigma_max,
     )

In [ ]:
# Display result
import matplotlib.pyplot as plt
sigma_global_opt = results_optim_global_sigma["solution"][-1].vec.FV().NumPy()[0]
print(f"Optimal global conductivity = {sigma_global_opt :.2e} S/m")
print(f"=> Joule losses = {results_optim_global_sigma["objective"][-1]:.2e} W/m")
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
ax1.plot(results_optim_global_sigma["objective"], 'b-')
ax2.semilogy(results_optim_global_sigma["criterion"], 'r-')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Joule losses (W/m)', color='b')
ax2.set_ylabel('Stop criterion', color='r')
plt.title(f"Convergence (global conductivity)")
plt.show()

In [ ]:
results_optim_global_sigma_DC = solve_magnetoharmonic(fes=fes,
                                reluctivity=nu,
                                conductivity=results_optim_global_sigma["solution"][-1],
                                magnetization=Mcplx,
                                frequency=1e-5,
                                supply=bundles,
                                   )

DC_losses_optim_global = joule_losses(results_optim_global_sigma_DC)
print(f"Global optim DC losses (phi = 150°): {DC_losses_optim_global:.2e} W/m")

_________
## 4) Conductivity optimization across each bundle

Since there is a lot of bundles, an exhaustive search becomes computionally too demanding. However, gradient descent scales well, without additional cost when computed with adjoint method.

In [ ]:
# Problem parameters
fes_sigma = L2(mesh, definedon = "slot(.*)_bundle.*")
x0 = GridFunction(fes_sigma)
x0.Set(sigma0)

# Derivative
def grad_joule_losses_bundle(result):
    fes_sigma = result["info"]["conductivity"].space
    dx_global = GridFunction(fes_sigma)
    dx_global.Set(1)
    df = d_joule_losses(result, dx_global)
    dfdx = CF(0)
    for bundle in result["info"]["supply"].keys():
        dx = GridFunction(fes_sigma)
        dx.Set(mesh.MaterialCF({bundle : 1}))
        dfdx += InnerProduct(df.vec, dx.vec) * copy(dx)
    grad_f = GridFunction(fes_sigma)
    grad_f.Set(dfdx.Compile())
    return grad_f

results_optim_bundle_sigma = gradient_descent(state=state_function,
                                 objective = objective_function,
                                 d_objective = grad_joule_losses_bundle,
                                 x0 = x0,
                                 x_min = sigma_min,
                                 x_max = sigma_max,
                                 descent=descent)

In [ ]:
result_optim_bundle = results_optim_bundle_sigma["solution"][-1] + mesh.MaterialCF({"slot.*_bundle.*" : 0}, default = np.nan)
Draw(result_optim_bundle, results_optim_bundle_sigma["solution"][-1].space.mesh,
     settings = {"Objects" : {"Wireframe" : False}, "Colormap" : {"ncolors" : 32}},
     filename = "sigma_optim_bundle.html",
     min = sigma_min, max = sigma_max
     )

In [ ]:
from utils.physics import average_property

sigma_bundle_optim = results_optim_bundle_sigma["solution"][-1]
state = results_optim_bundle_sigma["state"]
dico_sigma_bundle_optim = {}

for bundle in state["info"]["supply"].keys():
    dico_sigma_bundle_optim[bundle] = average_property(sigma_bundle_optim, state, zone = bundle)


fig, ax = plt.subplots(figsize=(5.5, 3.5))
x = np.arange(0, bundles_per_half_slot)
width = 0.3
rects1 = ax.bar(x - width/2, [dico_sigma_bundle_optim["slot11_bundle"+str(i)] for i in range(bundles_per_half_slot)] , width, label = "k={1,2,3,4,5}")
rects1 = ax.bar(x + width/2, [dico_sigma_bundle_optim["slot12_bundle"+str(i)] for i in range(bundles_per_half_slot)] , width, label = "k={6,7,8,9,10}")
    
# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_ylabel('Conductivity (S/m)')
#ax.set_title('Optimal conductivity per half slot and bundle')
ax.set_xticks(x) ; plt.grid()
ax.set_xlabel("Conductors sorted according to the distance to the air gap")
ax.legend()
plt.tight_layout()
plt.savefig("conductivity_optim_bundle.pdf")
plt.show()
print(f"=> Joule losses = {results_optim_bundle_sigma["objective"][-1]:.2e} W/m")

In [ ]:
# Display result
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
ax1.plot(results_optim_bundle_sigma["objective"], 'b-')
ax2.semilogy(results_optim_bundle_sigma["criterion"], 'r-')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Joule losses (W/m)', color='b')
ax2.set_ylabel('Stop criterion', color='r')
plt.title(f"Convergence to sigma = {conductivity_list[-1]:.2e} S/m, Pj = {joule_losses_list[-1]:.2e} W/m")
plt.show()

In [ ]:
results_optim_bundle_sigma_DC = solve_magnetoharmonic(fes=fes,
                                reluctivity=nu,
                                conductivity=results_optim_bundle_sigma["solution"][-1],
                                magnetization=Mcplx,
                                frequency=1e-5,
                                supply=bundles,
                                   )

DC_losses_optim_bundle = joule_losses(results_optim_bundle_sigma_DC)
print(f"Bundle optim DC losses (phi = 150°): {DC_losses_optim_bundle:.2e} W/m")

_________
## 4) Inhomogeneous conductivity optimization

To explore all the potential of additive manufacturing, we can explore the optimal conductivity distribution.

In [ ]:
# Problem parameters
fes_sigma = H1(mesh, order = 1, definedon = "slot(.*)_bundle.*")
x0 = GridFunction(fes_sigma)
x0.Set(sigma0)

# Derivative
def grad_joule_losses_bundle(result):
    fes_sigma = result["info"]["conductivity"].space
    dx_global = GridFunction(fes_sigma)
    dx_global.Set(1)
    df = d_joule_losses(result, dx_global)
    grad_f = GridFunction(fes_sigma)
    grad_f.vec.data = df.vec
    return grad_f

results_optim_topo_sigma = gradient_descent(state=state_function,
                                 objective = objective_function,
                                 d_objective = grad_joule_losses_bundle,
                                 x0 = x0,
                                 x_min = sigma_min,
                                 x_max = sigma_max,
                                 descent=descent)

In [ ]:
# Display result
fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
ax1.plot(results_optim_topo_sigma["objective"], 'b-')
ax2.semilogy(results_optim_topo_sigma["criterion"], 'r-')
ax1.set_xlabel('Iteration')
ax1.set_ylabel('Joule losses (W/m)', color='b')
ax2.set_ylabel('Stop criterion', color='r')
plt.title(f"Convergence topology optimization")
plt.show()

In [ ]:
print(f"=> Joule losses = {results_optim_topo_sigma["objective"][-1]:.2e} W/m")

In [ ]:
result_optim_topo = results_optim_topo_sigma["solution"][-1] + mesh.MaterialCF({"slot.*_bundle.*" : 0}, default = np.nan)
Draw(result_optim_topo, results_optim_topo_sigma["solution"][-1].space.mesh,
     settings = {"Objects" : {"Wireframe" : False}},
     filename = "sigma_optim_topo.html"
     )

In [ ]:
results_optim_topo_sigma_DC = solve_magnetoharmonic(fes=fes,
                                reluctivity=nu,
                                conductivity=results_optim_topo_sigma["solution"][-1],
                                magnetization=Mcplx,
                                frequency=1e-5,
                                supply=bundles,
                                   )

DC_losses_optim_topo = joule_losses(results_optim_topo_sigma_DC)
print(f"Topo optim DC losses (phi = 150°): {DC_losses_optim_topo:.2e} W/m")

In [ ]:
Irms